# Building a GPT-Style Language Model from Scratch

This educational implementation assembles a compact GPT-style transformer from text preparation through pretraining and task-specific fine-tuning. The focus is executable components, tensor interfaces, and the engineering decisions that connect them. It favors inspectability over production scale or throughput.

> **Implementation provenance.** Substantial implementation code is adapted from or closely follows Sebastian Raschka's [`LLMs-from-scratch`](https://github.com/rasbt/LLMs-from-scratch) repository at revision `33f5b246766464910accf1c70e668811cfc4bf08`. Local modifications exist. Attribution and third-party terms are documented in [`../THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md). Book prose and book images are not redistributed here.

## Contents

1. [Project Goal and Scope](#1-project-goal-and-scope)
2. [Text Preparation and Tokenization](#2-text-preparation-and-tokenization)
3. [Embeddings](#3-embeddings)
4. [Attention Mechanism](#4-attention-mechanism)
5. [GPT Architecture](#5-gpt-architecture)
6. [Language-Model Training](#6-language-model-training)
7. [Loading Pretrained GPT-2 Weights](#7-loading-pretrained-gpt-2-weights)
8. [Classification Fine-Tuning](#8-classification-fine-tuning)
9. [Instruction Fine-Tuning](#9-instruction-fine-tuning)
10. [Key Takeaways](#10-key-takeaways)
11. [Limitations and Next Steps](#11-limitations-and-next-steps)

## 1. Project Goal and Scope

The notebook implements the data path, attention modules, transformer backbone, optimization helpers, generation controls, and two fine-tuning workflows in plain PyTorch. Intermediate checks expose shapes and state transitions before components are composed. This is not a novel architecture or a claim that adapted code was independently invented; its portfolio value is end-to-end integration and technical reasoning.

## 2. Text Preparation and Tokenization

The pipeline begins with the tracked transcription of Edith Wharton's “The Verdict,” whose immediate provenance is documented in the project notices. The first cells load its UTF-8 character stream for controlled tokenizer experiments.

> **Adapted implementation.** The simple tokenizers, `GPTDatasetV1`, and dataloader helper closely follow the pinned Raschka implementation. Source mapping and applicable terms are in [`../THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

> **Runtime paths.** Paths are resolved from `PROJECT_ROOT`; tracked inputs remain in `data/`, while downloads and generated artifacts go to the ignored `.runtime/` directory. Launching Jupyter from the project root is recommended.

In [ ]:
import sys
from pathlib import Path


PROJECT_MARKERS = (
    Path("data/the-verdict.txt"),
    Path("notebooks/llm-from-scratch.ipynb"),
    Path("THIRD_PARTY_NOTICES.md"),
)


def resolve_project_root(start_path=None):
    start = Path.cwd() if start_path is None else Path(start_path)
    start = start.resolve()
    candidates = (
        start,
        start / "projects" / "llm-from-scratch",
        start.parent,
    )

    for candidate in candidates:
        if all((candidate / marker).is_file() for marker in PROJECT_MARKERS):
            return candidate

    raise RuntimeError(
        "Could not locate the llm-from-scratch project. Launch Jupyter from "
        "the repository root, project root, or notebook directory."
    )


PROJECT_ROOT = resolve_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"
RUNTIME_DIR = PROJECT_ROOT / ".runtime"
DOWNLOAD_DIR = RUNTIME_DIR / "downloads"
ARTIFACT_DIR = RUNTIME_DIR / "generated"
CHECKPOINT_DIR = RUNTIME_DIR / "checkpoints"
PLOT_DIR = RUNTIME_DIR / "plots"
DATASET_DIR = RUNTIME_DIR / "datasets"
GPT2_DIR = RUNTIME_DIR / "gpt2"

for directory in (
    RUNTIME_DIR, DOWNLOAD_DIR, ARTIFACT_DIR, CHECKPOINT_DIR,
    PLOT_DIR, DATASET_DIR, GPT2_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = DATA_DIR / "the-verdict.txt"
MODEL_OPTIMIZER_PATH = CHECKPOINT_DIR / "model_and_optimizer.pth"
GPT_DOWNLOAD_PATH = DOWNLOAD_DIR / "gpt_download.py"
SMS_ZIP_PATH = DOWNLOAD_DIR / "sms_spam_collection.zip"
SMS_EXTRACTED_DIR = DATASET_DIR / "sms_spam_collection"
TRAIN_CSV_PATH = DATASET_DIR / "train.csv"
VALIDATION_CSV_PATH = DATASET_DIR / "validation.csv"
TEST_CSV_PATH = DATASET_DIR / "test.csv"
REVIEW_CLASSIFIER_PATH = CHECKPOINT_DIR / "review_classifier.pth"
INSTRUCTION_DATA_PATH = DATASET_DIR / "instruction-data.json"
INSTRUCTION_RESPONSES_PATH = ARTIFACT_DIR / "instruction-data-with-response.json"
SFT_CHECKPOINT_PATH = CHECKPOINT_DIR / "gpt2-small124M-sft.pth"

with CORPUS_PATH.open("r", encoding="utf-8") as file:
    raw_text = file.read()

print("Total number of characters:", len(raw_text))
print(raw_text[:99])


### 2.1 Basic tokenization and vocabulary

A small regular-expression tokenizer exposes mechanics hidden by library tokenizers. It separates punctuation, removes empty whitespace fragments, and yields a deterministic sequence. The sorted set of observed tokens becomes a reproducible token-to-ID vocabulary; ID magnitude carries no semantic meaning.

In [ ]:
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(preprocessed[:100])


In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)


In [ ]:
vocab = {token: integer for integer, token in enumerate(all_words)}

for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 51:
        break


`SimpleTokenizerV1` stores lookup tables in both directions. `encode` applies the vocabulary's splitting rule; `decode` reconstructs tokens and repairs punctuation spacing. Its deliberate limitation is failure on unseen tokens.

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[token] for token in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join(self.int_to_str[token_id] for token_id in ids)
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text


In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = '''"It's the last he painted, you know," Mrs. Gisburn said with pardonable
pride.'''

ids = tokenizer.encode(text)
print(ids)


In [ ]:
print(tokenizer.decode(ids))


In [ ]:
text = "Hello, do you like tea?"

try:
    print(tokenizer.encode(text))
except KeyError as error:
    print(f"Unknown token: {error}")


### 2.2 Boundaries and unknown values

`<|endoftext|>` marks boundaries between independent text segments, while `<|unk|>` gives out-of-vocabulary items a defined ID. These reserved values extend the corpus-derived vocabulary without pretending unknown spellings are recoverable.

In [ ]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: integer for integer, token in enumerate(all_tokens)}

print(len(vocab))


In [ ]:
for item in list(vocab.items())[-5:]:
    print(item)


`SimpleTokenizerV2` substitutes the unknown marker before lookup, preventing the dictionary error produced by version 1. Decoding recovers only the marker, so this remains a teaching tokenizer rather than a general text encoding.

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>"
            for item in preprocessed
        ]
        ids = [self.str_to_int[token] for token in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join(self.int_to_str[token_id] for token_id in ids)
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text


In [ ]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))

print(text)


In [ ]:
tokenizer = SimpleTokenizerV2(vocab)
encoded_text = tokenizer.encode(text)

print(encoded_text)


In [ ]:
print(tokenizer.decode(encoded_text))


### 2.3 GPT-2 byte-pair encoding

The notebook then switches to `tiktoken` and the GPT-2 vocabulary. Subword encoding can represent unfamiliar strings from smaller units and supplies the 50,257-token ID space expected by the later model configuration.

In [ ]:
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))


In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")


In [ ]:
text = '''Hello, do you like tea? <|endoftext|> In the sunlit terraces of
someunknownPlace.'''

integers = tokenizer.encode(
    text,
    allowed_special={"<|endoftext|>"}
)

print(integers)


In [ ]:
decoded_text = tokenizer.decode(integers)
print(decoded_text)


The per-token inspection checks how a string is partitioned, while complete-sequence decoding verifies round-trip behavior. Production logic should decode the sequence as a whole because byte boundaries need not align with individual token boundaries.

In [ ]:
special_text = "Akwirw ier"
special_integers = tokenizer.encode(special_text)

print(special_integers)

for token_id in special_integers:
    print(token_id, repr(tokenizer.decode([token_id])))


In [ ]:
decoded_special_text = tokenizer.decode(special_integers)
print(decoded_special_text)


### 2.4 Context windows

Training examples come from a fixed-width window over encoded text. `max_length` sets the tokens visible per sample; `stride` sets the next starting position. A smaller stride creates more overlap and more strongly correlated windows.

In [ ]:
with CORPUS_PATH.open("r", encoding="utf-8") as file:
    raw_text = file.read()

enc_text = tokenizer.encode(raw_text)

print(len(enc_text))


In [ ]:
enc_sample = enc_text[50:]


Next-token supervision is produced by shifting one sequence:

\[
x=(t_0,\ldots,t_{n-1}),\qquad y=(t_1,\ldots,t_n).
\]

Every input position is therefore paired with the token immediately following it.

In [ ]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f"x: {x}")
print(f"y: {y}")


In [ ]:
# convert the token IDs into text
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

In [ ]:
# We've now created the input-target pairs that we can turn into use for the LLM training in
# upcoming chapters.
# There's only one more task before we can turn the tokens into embeddings, as we
# mentioned at the beginning of this chapter: implementing an efficient data loader that
# iterates over the input dataset and returns the inputs and targets as PyTorch tensors, which
# can be thought of as multidimensional arrays.
# In particular, we are interested in returning two tensors: an input tensor containing the
# text that the LLM sees and a target tensor that includes the targets for the LLM to predict

In [ ]:
# To implement efficient data loaders, we collect the inputs in a tensor, x, where each row
# represents one input context. A second tensor, y, contains the corresponding prediction targets (next words),
# which are created by shifting the input by one position.

In [ ]:
#  To implement efficient data loaders, we collect the inputs in a tensor, x, where each row
# represents one input context. A second tensor, y, contains the corresponding prediction targets (next words),
# which are created by shifting the input by one position.
# For the efficient data loader implementation, we will use PyTorch's built-in Dataset and
# DataLoader classes. 

### 2.5 Dataset and dataloader

`GPTDatasetV1` tokenizes once, extracts aligned windows, and returns `torch.long` input and target tensors. Precomputing windows is reasonable for this small corpus, though a streaming design would scale better.

In [ ]:
from src.llm_from_scratch.data import GPTDatasetV1


`create_dataloader_v1` makes batch size, context width, stride, shuffling, incomplete-batch handling, and worker count explicit. These settings control memory use, sample overlap, and evaluation determinism.

In [ ]:
from src.llm_from_scratch.data import create_dataloader_v1


### 2.6 Retained PyTorch mechanics checks

The next compact code experiments verify tensor creation, reshaping, matrix multiplication, and automatic differentiation before those operations appear inside the transformer. They are retained as executable prerequisites, not as a general PyTorch tutorial.

In [ ]:
import torch
# create a 0D tensor (scalar) from a Python integer
tensor0d = torch.tensor(1)
# create a 1D tensor (vector) from a Python list
tensor1d = torch.tensor([1, 2, 3])
# create a 2D tensor from a nested Python list
tensor2d = torch.tensor([[1, 2], [3, 4]])

#create a 3D tensor from a nested Python list
tensor3d = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])



In [ ]:
tensor1d = torch.tensor([1, 2, 3])
print(tensor1d.dtype)
# this prints : torch.164


In [ ]:
floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

In [ ]:
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

In [ ]:
tensor2d = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(tensor2d)

In [ ]:
print(tensor2d.shape)

In [ ]:
print(tensor2d.reshape(3, 2))

In [ ]:
print(tensor2d.view(3, 2))

In [ ]:
print(tensor2d.T)

In [ ]:
print(tensor2d.matmul(tensor2d.T))

In [ ]:
print(tensor2d @ tensor2d.T)

In [ ]:
# This import statement is a common convention in PyTorch to prevent long lines of code
import torch.nn.functional as F

# true label
y = torch.tensor([1.0])
# input feature
x1 = torch.tensor([1.1])
# weight parameter
w1 = torch.tensor([2.2])
# bias unit
b = torch.tensor([0.0])
# net input
z = x1 * w1 + b
# activation & output
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

In [ ]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

# By default, PyTorch destroys the computation graph after calculating the gradients to free memory.
# However, since we are going to reuse this computation graph shortly, we set retain_graph=True so that it stays
# in memory.

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

In [ ]:
# Let`s show the resulting values of the loss with respect to the model`s parameters:
print(grad_L_w1)
print(grad_L_b)

In [ ]:
loss.backward()
print(w1.grad)
print(b.grad)

### 2.7 Inspecting batches

Small loader runs make the input-target offset and leading batch dimension concrete. Adjacent windows overlap when stride is smaller than context length; increasing stride reduces overlap but produces fewer samples.

In [ ]:
import tiktoken

# We test the dataloader with a batch of the size 1 for an LLM with a context size of 4 

with CORPUS_PATH.open("r", encoding="utf-8") as f:
    raw_text = f.read()
    
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
# convert dataloader into a Python iterator to fetch the next entry via Python's built-in next() function
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
# to illustrate the meaning of stride = 1 we fetch another batch from the dataset
second_batch = next(data_iter)
print(second_batch)

Changing `max_length` and `stride` separates model context from sampling density. The former controls tokens per example, while the latter controls how far the corpus cursor advances.

In [ ]:
# To understand it better we experiment with the settings
# max_length -> determines how many token IDs the model "sees" at once. It directly defines the length of the tensors in the batch
# max_length=5 -> input and output tensors have exactly 5 IDs

import tiktoken

with CORPUS_PATH.open("r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader_test_1 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=5, stride=1, shuffle=False)
data_iter = iter(dataloader_test_1)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
# max_length=6 -> input and output tensors have exactly 6 IDs

import tiktoken

with CORPUS_PATH.open("r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader_test_2 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=6, stride=1, shuffle=False)
data_iter = iter(dataloader_test_2)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
# the stride parameter determines how many positions the sliding window shifts to the right to generate the next batch.
# stride=2 -> The window skips one position and moves 2 forward


import tiktoken

with CORPUS_PATH.open("r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader_test_3 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=2, shuffle=False)
data_iter = iter(dataloader_test_3)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
# stride=3 -> The window skips one position and moves 3 forward


import tiktoken

with CORPUS_PATH.open("r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader_test_4 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=3, shuffle=False)
data_iter = iter(dataloader_test_4)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
import tiktoken

with CORPUS_PATH.open("r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader_test_5 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=8, stride=5, shuffle=False)
data_iter = iter(dataloader_test_5)
first_batch = next(data_iter)
print(first_batch)

At the end of this stage, both inputs and targets have shape `(batch, tokens)`. They contain categorical IDs and are ready for embedding lookup.

In [ ]:
# Let`s have a look at how we can use the data loader to sample with a batch size greater than 1:
# batch_size=8 , max_length=4 , stride=4
# This utilizes the full dataset, and no word is skipped while avoiding any overlap ( wich could lead to increased overfitting)
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)
# This prints the following:
# Inputs:
# tensor([[40, 367, 2885, 1464],
#       [ 1807, 3619, 402, 271],
#       [ 10899, 2138, 257, 7026], 
#       [ 15632, 438, 2016, 257],
#       [ 922, 5891, 1576, 438],
#       [ 568, 340, 373, 645],
#       [ 1049, 5975, 284, 502],
#       [ 284, 3285, 326, 11]])

# Targets:
# tensor([[ 367, 2885, 1464,1807],
#         [ 3619, 402, 271, 10899],
#        [ 2138, 257, 7026, 15632],
#        [ 438, 2016, 257, 922],
#        [ 5891, 1576, 438, 568],
#        [ 340, 373, 645, 1049],
#        [ 5975, 284, 502, 284],
#        [ 3285, 326, 11, 287]])

## 3. Embeddings

A token embedding is a trainable lookup table

\[
E\in\mathbb{R}^{V\times d},
\]

where `V` is vocabulary size and `d` is embedding width. Indexing it appends `d` as the final tensor dimension.

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

### 3.1 Token lookup

The six-token demonstration isolates lookup behavior. An ID selects one row; a sequence selects multiple rows in order. Initial values are random parameters, not fixed semantic features.

In [ ]:
vocab_size = 6
output_dim = 3

`vocab_size` fixes the row count and `output_dim` fixes each row's width. Their product is the parameter count of this demonstration table.

In [ ]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

A fixed seed makes initialization repeatable. Repeated use of the same ID selects the same vector until optimization changes the embedding weights.

In [ ]:
print(embedding_layer(torch.tensor([3])))

Embedding a sequence produces `(tokens, embedding_dim)`; embedding a batch produces `(batch, tokens, embedding_dim)`. PyTorch preserves all input dimensions and adds the lookup width.

In [ ]:
print(embedding_layer(input_ids))

### 3.2 Position lookup

Token lookup alone is insensitive to order. A second table indexed from zero to `context_length - 1` supplies a learned vector for each position in the active window.

In [ ]:
# Size of the Vocabulary is 50257
vocab_size = 50257
# embeds each token in each batch  into a 256-dimensional vector
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

For input shape `(batch, tokens)`, token vectors have shape `(batch, tokens, embedding_dim)` and position vectors have shape `(tokens, embedding_dim)`.

In [ ]:
# First we instantiate the data loader from earlier (data_loader_v1) -> Data sampling with a sliding window

max_length = 4
dataloader = create_dataloader_v1(
raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

Broadcasting adds the position matrix to every sample without changing the result shape. The combined `(batch, tokens, embedding_dim)` tensor is the interface consumed by attention.

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

This establishes an architectural invariant: later modules may split, expand, or mix features internally, but transformer blocks preserve the external embedding width.

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

## 4. Attention Mechanism

### 4.1 Simplified Self-Attention

The first implementation uses embedded tokens directly. Dot products measure alignment, row-wise normalization produces weights, and weighted aggregation produces one context vector per token.

> **Adapted implementation.** The attention progression and module implementations closely follow Raschka's pinned revision `33f5b246766464910accf1c70e668811cfc4bf08`. Detailed paths and terms are in [`../THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

In [ ]:
import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89],  # Your (x^1)
[0.55, 0.87, 0.66],   # journey (x^2)
[0.57, 0.85, 0.64],   # starts (x^3)
[0.22, 0.58, 0.33],   # with (x^4)
[0.77, 0.25, 0.10],   # one (x^5)
[0.05, 0.80, 0.55]]   # step (x^6)
)

For one query, dot products against all token vectors produce one score per source position. Raw scores indicate relative alignment but are not yet a probability distribution.

Softmax converts scores into non-negative weights that sum to one along the token axis. Each query row therefore distributes attention independently.

In [ ]:
#The second input token serves as the query
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

In [ ]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

The context vector is the weighted sum of source vectors. It keeps the input feature width while incorporating information from the complete sequence.

In [ ]:
query = inputs[1] # 2nd input token is the query
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

Matrix multiplication replaces nested loops: `X @ X.T` computes every token-pair score simultaneously.

In [ ]:
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

Normalizing the last axis treats rows as receiving queries and columns as contributing source positions.

In [ ]:
attn_scores = inputs @ inputs.T
print(attn_scores)

In [ ]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

In [ ]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)
print("All row sums:", attn_weights.sum(dim=-1))

Multiplying the weight matrix by `X` yields all context vectors in one operation. The score-normalize-aggregate pattern remains in every later attention module.

In [ ]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

In [ ]:
print("Previous 2nd context vector:", context_vec_2)

### 4.2 Trainable Query, Key, and Value Projections

Trainable projections give each input three roles:

\[
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.
\]

In [ ]:
x_2 = inputs[1]           # The second input element
d_in = inputs.shape[1]    # The input embedding size, d=3
d_out = 2                 # The output embedding size, d_out=2

The small demonstration projects three-dimensional inputs into a narrow space for inspection. In the full model, the combined projection width matches the external embedding dimension.

In [ ]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

Projection matrices are persistent learned parameters. Attention weights are input-dependent activations recomputed during each forward pass.

In [ ]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

A single query still needs all keys and values: it compares against every key, then uses the normalized row to mix every value.

In [ ]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

Scaled dot-product attention uses

\[
A=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right),\qquad Z=AV.
\]

Scaling prevents score magnitude from growing unchecked with key width.

In [ ]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

In [ ]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

In [ ]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

Projecting the full input matrix creates `Q`, `K`, and `V` for every position, producing one output row for every input token.

In [ ]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

`SelfAttention_v1` mirrors the equations with explicit parameter matrices. It is easy to inspect because each algebraic step is visible in `forward`.

In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

        
    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

`SelfAttention_v2` uses bias-free `nn.Linear` projections. Tensor flow is unchanged, while parameter registration and initialization follow standard PyTorch layer behavior.

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

In [ ]:
# Input and output dimensions
d_in = inputs.shape[1]
d_out = 2

# Create both objects
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(
    d_in,
    d_out,
    qkv_bias=False
)

# Copy the weights from v2 to v1
# nn.Linear stores the weights as [d_out, d_in],
# whereas v1 expects [d_in, d_out].
with torch.no_grad():
    sa_v1.W_query.copy_(sa_v2.W_query.weight.T)
    sa_v1.W_key.copy_(sa_v2.W_key.weight.T)
    sa_v1.W_value.copy_(sa_v2.W_value.weight.T)

# Run both models with the same input
output_v1 = sa_v1(inputs)
output_v2 = sa_v2(inputs)

print("Output SelfAttention_v1:")
print(output_v1)

print("\nOutput SelfAttention_v2:")
print(output_v2)

print("\nOutputs equal:")
print(torch.allclose(output_v1, output_v2))

### 4.3 Causal Attention

Autoregressive prediction must hide future positions. A triangular score mask supplies negative infinity for prohibited pairs:

\[
A=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right).
\]

Row `i` may use positions up to `i`; columns to its right must receive zero probability after normalization. The explicit mask checks this orientation.

In [ ]:
# Reuse the query and key weight matrices of the SelfAttention_v2 object from the previous section for
#convenience
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

In [ ]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

In [ ]:
row_sums = masked_simple.sum(dim=1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

Masking scores before softmax performs exclusion and renormalization together. Negative infinity becomes zero probability without a second normalization pass.

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

In [ ]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

Attention dropout removes some normalized connections during training and rescales survivors. Evaluation mode disables this randomness.

In [ ]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) #  choose a dropout rate of 50%
example = torch.ones(6, 6) # we create a matrix of 1`s
print(dropout(example))

In [ ]:
torch.manual_seed(123)
print(dropout(attn_weights))

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # A 2 inputs with 6 tokens each, and each token has embedding dimension 3

`CausalAttention` accepts `(batch, tokens, input_dim)` and stores its triangular mask as a registered, non-trainable buffer that moves with the module.

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        #  Compared to the previous SelfAttention_v1 class, we added a dropout layer
        self.dropout = nn.Dropout(dropout) 
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length),
            diagonal=1)
            # The register_buffer call is also a new addition (more information is provided in the following text)
        )
 
    def forward(self, x):
        # We transpose dimensions 1 and 2, keeping the batch dimension at the first position (0)
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        # We transpose dimensions 1 and 2, keeping the batch dimension at the first position (0)
        attn_scores = queries @ keys.transpose(1, 2)
        # In PyTorch, operations with a trailing underscore are performed in-place, avoiding unnecessary memory
        # copies
        attn_scores.masked_fill_(
         self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

### 4.4 Multi-Head Attention

The wrapper version concatenates several causal-attention modules. It makes parallel representation subspaces explicit, though it is less efficient than fused projections.

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                dropout, num_heads, qkv_bias=False):
            super().__init__()
            self.heads = nn.ModuleList(
                [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)]
            )
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:",context_vecs.shape)

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 1
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:",context_vecs.shape)

`MultiHeadAttention` projects all heads together, then reshapes the feature axis so

\[
\text{head\_dim}=\frac{\text{embedding\_dim}}{\text{num\_heads}}.
\]

In [ ]:
from src.llm_from_scratch.model import MultiHeadAttention


Transposing to `(batch, heads, tokens, head_dim)` enables batched score calculation. Contexts are transposed back and made contiguous before heads are combined.

The output projection mixes concatenated heads while preserving `(batch, tokens, embedding_dim)`, allowing attention modules and transformer blocks to stack.

In [ ]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

For GPT-2-small dimensions, 768 features divided across 12 heads gives 64 features per head. An independently generated tensor-flow visualization may be added later.

In [ ]:
torch.manual_seed(123)

d_in = 768
d_out = 768
context_length = 1024
num_heads = 12
dropout = 0.0

mha = MultiHeadAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=dropout,
    num_heads=num_heads,
    qkv_bias=False
)

print(mha)
print("Dimension pro Attention Head:", mha.head_dim)

## 5. GPT Architecture

### 5.1 Model Configuration

One dictionary centralizes vocabulary size, context length, embedding width, heads, blocks, dropout, and projection-bias behavior. Passing it through the module tree keeps dimensions consistent.

In [ ]:
from src.llm_from_scratch.config import GPT_CONFIG_124M


The dummy backbone checks the high-level path before real components exist: embeddings are added, dropout is applied, repeated blocks transform the sequence, and an output head emits vocabulary logits.

In [ ]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]) # Use a placeholder for TransformerBlock
        self.final_norm = DummyLayerNorm(cfg["emb_dim"]) # Use a placeholder for LayerNorm
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )
        
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

        
class DummyTransformerBlock(nn.Module): # A simple placeholder class that will be replaced by a real TransformerBlock later
    def __init__(self, cfg):
        super().__init__()


    def forward(self, x): # This block does nothing and just returns its input.
        return x

class DummyLayerNorm(nn.Module): # A simple placeholder class that will be replaced by a real TransformerBlock later
    def __init__(self, normalized_shape, eps=1e-5): # The parameters here are just to mimic the LayerNorm interface.
        super().__init__()

    def forward(self, x):
        return x

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

In [ ]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

### 5.2 Layer Normalization and GELU

Layer normalization standardizes the final feature axis independently for each token while preserving batch, sequence, and embedding dimensions.

In [ ]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

The small network computes mean and variance along the final axis. Retaining that axis during reduction allows clean broadcasting in normalization.

In [ ]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

Learned scale and shift restore representational flexibility after normalization, and epsilon protects the denominator from numerical instability.

In [ ]:
out_norm = (out - mean) / torch.sqrt(var)
mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("Normalized layer outputs:\n", out_norm)
print("Mean:\n", mean)
print("Variance:\n", var)

> **Adapted implementation.** `LayerNorm`, `GELU`, `FeedForward`, `TransformerBlock`, `GPTModel`, and the basic generation helper closely follow the pinned Raschka architecture code. See [`../THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

In [ ]:
from src.llm_from_scratch.model import LayerNorm


In [ ]:
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

The retained multilayer-perceptron check demonstrates nested module registration. The GPT feed-forward component uses the same composition pattern with configuration-driven dimensions.

In [ ]:
class NeuralNetwork(torch.nn.Module):
    # It's useful to code the number of inputs and outputs as variables to reuse the same code for datasets with
    # different numbers of features and classes.
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        
        self.layers = torch.nn.Sequential(
        # 1st hidden layer
        # The Linear layer takes the number of input and output nodes as arguments.
        torch.nn.Linear(num_inputs, 30),
        # Nonlinear activation functions are placed between the hidden layers.
        torch.nn.ReLU(),
            
        # 2nd hidden layer
        # The number of output nodes of one hidden layer has to match the number of inputs of the next layer.
        torch.nn.Linear(30, 20),
        torch.nn.ReLU(),
            
        # output layer
        torch.nn.Linear(20, num_outputs),
    )
    def forward(self, x):
        logits = self.layers(x)
        # The outputs of the last layer are called logits.
        return logits

In [ ]:
model_per = NeuralNetwork(50, 3)
print(model_per)

In [ ]:
num_params = sum(p.numel() for p in model_per.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

In [ ]:
print(model_per.layers[0].weight)

In [ ]:
torch.manual_seed(123)
model_per = NeuralNetwork(50, 3)
print(model_per.layers[0].weight)

The implemented GELU approximation is

\[
\operatorname{GELU}(x)\approx\tfrac12x\left(1+\tanh\!\left[\sqrt{\tfrac{2}{\pi}}\left(x+0.044715x^3\right)\right]\right).
\]

In [ ]:
from src.llm_from_scratch.model import GELU


In [ ]:
import matplotlib.pyplot as plt
gelu, relu = GELU(), nn.ReLU()
# # A Create 100 sample data points in the range -3 to 3
x = torch.linspace(-3, 3, 100)
 
y_gelu, y_relu = gelu(x), relu(x)
plt.figure(figsize=(8, 3))
for i, (y, label) in enumerate(zip([y_gelu, y_relu], ["GELU", "ReLU"]), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{label} activation function")
    plt.xlabel("x")
    plt.ylabel(f"{label}(x)")
    plt.grid(True)
plt.tight_layout()
plt.show()

### 5.3 Feed-Forward Network and Residual Connections

`FeedForward` expands each token from `emb_dim` to `4 * emb_dim`, applies GELU, and projects back. It mixes features within tokens, not positions across the sequence.

In [ ]:
from src.llm_from_scratch.model import FeedForward


In [ ]:
ffn = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2, 3, 768) # Create a sample input with batch dimensions 2
out = ffn(x)
print(out.shape)

The residual check explores direct identity paths. The pattern used later is

\[
x_{\text{out}}=x+F(\operatorname{LN}(x)).
\]

In [ ]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            # Implement 5 layers
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
        ])
    def forward(self, x):
        for layer in self.layers:
            # Compute the output of the current layer
            layer_output = layer(x)
            # Check if shortcut can be applied
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x

In [ ]:
layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])
torch.manual_seed(123) # specify random seed for the initial weights for reproducibility
model_without_shortcut = ExampleDeepNeuralNetwork(
layer_sizes, use_shortcut=False
)

The gradient probe runs one backward pass and reports layer-wise gradient magnitude. It checks residual behavior; it is not a training benchmark.

In [ ]:
def print_gradients(model, x):
    # Forward pass
    output = model(x)
    target = torch.tensor([[0.]])
    
    # Calculate loss based on how close the target
    # and output are
    loss = nn.MSELoss()
    loss = loss(output, target)
    
    # Backward pass to calculate the gradients
    loss.backward()
    
    for name, param in model.named_parameters():
        if 'weight' in name:
            # Print the mean absolute gradient of the weights
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [ ]:
print_gradients(model_without_shortcut, sample_input)

In [ ]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

### 5.4 Transformer Block

`TransformerBlock` has two pre-normalized residual branches: masked multi-head attention and a position-wise feed-forward network. Dropout is applied before each residual addition.

In [ ]:
from src.llm_from_scratch.model import TransformerBlock


A block accepts and returns `(batch, tokens, emb_dim)`. Shape preservation permits repeated blocks inside `nn.Sequential`.

In [ ]:
torch.manual_seed(123)
x = torch.rand(2, 4, 768) # Create sample input of shape [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

### 5.5 Complete GPT Model

`GPTModel` combines token and position embeddings, embedding dropout, stacked transformer blocks, final normalization, and a vocabulary projection.

In [ ]:
from src.llm_from_scratch.model import GPTModel


The forward result has shape `(batch, tokens, vocab_size)`. Its final axis contains one unnormalized score for every possible next token at every position.

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Parameter counting is a compatibility check, not a performance claim. Vocabulary-sized embedding and output matrices account for much of the total.

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

In [ ]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

The weight-tying comparison explains why nominal GPT-2 counts differ from a model with separate input embeddings and output projection parameters.

In [ ]:
total_params_gpt2 = total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

Component counts expose capacity allocation: attention mixes positions, while the expanded feed-forward layers contribute a large share of each block's parameters.

In [ ]:
block = TransformerBlock(GPT_CONFIG_124M)

ff_params = sum(
    parameter.numel()
    for parameter in block.ff.parameters()
)

attention_params = sum(
    parameter.numel()
    for parameter in block.att.parameters()
)

print(f"Feed-forward parameters: {ff_params:,}")
print(f"Multi-head attention parameters: {attention_params:,}")
print(f"Ratio FF / attention: {ff_params / attention_params:.2f}")

In [ ]:
print("FEED-FORWARD MODULE")
for name, parameter in block.ff.named_parameters():
    print(
        f"{name:25} "
        f"shape={str(tuple(parameter.shape)):18} "
        f"parameters={parameter.numel():,}"
    )

print("\nMULTI-HEAD ATTENTION MODULE")
for name, parameter in block.att.named_parameters():
    print(
        f"{name:25} "
        f"shape={str(tuple(parameter.shape)):18} "
        f"parameters={parameter.numel():,}"
    )

The parameter-memory estimate excludes gradients, optimizer state, activations, temporary buffers, and framework overhead; actual training demand is higher.

In [ ]:
total_size_bytes = total_params * 4 # Calculate the total size in bytes (assuming float32, 4 bytes per parameter)
total_size_mb = total_size_bytes / (1024 * 1024) # Convert to megabytes
print(f"Total size of the model: {total_size_mb:.2f} MB")

In [ ]:
# PC zu schwach deswegen auskommentiert

"""


BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.1,
    "qkv_bias": False
}

GPT_CONFIG_MEDIUM = {
    **BASE_CONFIG,
    "emb_dim": 1024,
    "n_layers": 24,
    "n_heads": 16
}

GPT_CONFIG_LARGE = {
    **BASE_CONFIG,
    "emb_dim": 1280,
    "n_layers": 36,
    "n_heads": 20
}

GPT_CONFIG_XL = {
    **BASE_CONFIG,
    "emb_dim": 1600,
    "n_layers": 48,
    "n_heads": 25
}

"""

In [ ]:
"""

import gc

model_configs = {
    "GPT-2 medium": GPT_CONFIG_MEDIUM,
    "GPT-2 large": GPT_CONFIG_LARGE,
    "GPT-2 XL": GPT_CONFIG_XL
}

for model_name, config in model_configs.items():
    model = GPTModel(config)

    # Count all parameters in the book's GPTModel implementation
    total_params = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    # Original GPT-2 shares the token embedding and output-head weights.
    # Subtract the separate output head to simulate this weight tying.
    output_head_params = sum(
        parameter.numel()
        for parameter in model.out_head.parameters()
    )

    tied_params = total_params - output_head_params

    print(f"\n{model_name}")
    print(f"Embedding dimension: {config['emb_dim']}")
    print(f"Transformer blocks: {config['n_layers']}")
    print(f"Attention heads: {config['n_heads']}")
    print(f"Parameters in GPTModel: {total_params:,}")
    print(f"Parameters with weight tying: {tied_params:,}")

    # Delete the model before constructing the next large model
    del model
    gc.collect()

"""

### 5.6 Autoregressive Generation

`generate_text_simple` crops the active context, reads final-position logits, selects the highest-scoring token, and appends it until the requested length is reached.

In [ ]:
from src.llm_from_scratch.training import generate_text_simple


In [ ]:
import tiktoken

start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
print("encoded_tensor.shape:", encoded_tensor.shape)

Evaluation mode and `torch.no_grad()` disable dropout and gradient recording, making greedy generation deterministic for fixed inputs and parameters.

In [ ]:
model.eval() # disable dropout since we are not training the model
out = generate_text_simple(
model=model,
idx=encoded_tensor,
max_new_tokens=6,
context_size=GPT_CONFIG_124M["context_length"]
)
print("Output:", out)
print("Output length:", len(out[0]))

The current model is randomly initialized, so decoded text is only an integration check until training or pretrained parameter loading occurs.

In [ ]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

## 6. Language-Model Training

### 6.1 Next-Token Loss

Training compares vocabulary logits at every position with the shifted target ID. The objective rewards parameter settings that assign greater probability to the observed continuation.

> **Adapted implementation.** Loss, evaluation, training, sampling, and generation helpers closely follow the pinned Raschka code. Source paths and terms are in [`../THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

In [ ]:
import torch

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256, # We shorten the context length from 1024 to 256 tokens
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1, #  It's possible and common to set dropout to 0
    "qkv_bias": False
}
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

Conversion helpers keep tokenizer logic at the boundary: prompts become batched ID tensors before evaluation, and generated tensors are decoded only for human inspection.

In [ ]:
from src.llm_from_scratch.tokenizer import text_to_token_ids, token_ids_to_text
import tiktoken

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)
print("Output text:
", token_ids_to_text(token_ids, tokenizer))

Model output has shape `(batch, tokens, vocab_size)` and targets have shape `(batch, tokens)`. The vocabulary axis is the class dimension for each token position.

In [ ]:
inputs = torch.tensor([[16833, 3626, 6100], # ["every effort moves",
                        [40, 1107, 588]]) # "I really like"]

# Matching these inputs, the `targets` contain the token IDs we aim for the model to
# produce:
targets = torch.tensor([[3626, 6100, 345], # [" effort moves you",
                        [107,588, 11311]]) # " really like chocolate"]

In [ ]:
with torch.no_grad():  # Disable gradient tracking since we are not training, yet
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1) # Probability of each token in vocabulary
print(probas.shape)

Argmax is useful for diagnostics, but training optimizes the complete logit distribution so gradients remain informative for every vocabulary candidate.

In [ ]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", token_ids)

In [ ]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

For `N` flattened positions, next-token cross-entropy is

\[
\mathcal{L}=-\frac1N\sum_{i=1}^{N}\log p_\theta(y_i\mid x_{\le i}).
\]

In [ ]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)
text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)

Negative mean log probability turns higher target probability into lower loss. PyTorch combines log-softmax and negative log-likelihood in one stable operation.

In [ ]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

In [ ]:
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

In [ ]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

Logits reshape to `(batch * tokens, vocab_size)` and targets to `(batch * tokens)`. Flattening changes enumeration, not prediction-target alignment.

In [ ]:
print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)

In [ ]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Perplexity is a monotonic view of average cross-entropy:

\[
\operatorname{PPL}=\exp(\mathcal{L}).
\]

No numerical value is asserted before reproducible execution.

In [ ]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

### 6.2 Training and Validation

The tracked corpus is split before window construction. Training batches shuffle; held-out validation batches use deterministic order and the same tokenizer and context width.

In [ ]:
with CORPUS_PATH.open("r", encoding="utf-8") as file:
    text_data = file.read()

Character count, token count, split fraction, context width, and stride determine the available batches. The corpus is intentionally small, so overfitting is expected.

In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
print("Characters:", total_characters)
print("Tokens:", total_tokens)

In [ ]:
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

In [ ]:
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)
    
print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

`calc_loss_batch` computes one device-aware loss. `calc_loss_loader` averages batches from a complete loader or a bounded subset.

In [ ]:
from src.llm_from_scratch.training import calc_loss_batch


In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader) #  Iterative over all batches if no fixed num_batches is specified
    else:
        num_batches = min(num_batches, len(data_loader)) #  Reduce the number of batches to match the total number of batches in the data loader if num_batches
                                                        # exceeds the number of batches in the data loader
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item() # Sum loss for each batch
        else:
            break
    return total_loss / num_batches # Average the loss over all batches


In [ ]:
# If you have a machine with a CUDA-supported GPU, the LLM will train on the GPU without making any
# changes to the code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
model.to(device)
# Disable gradient tracking for efficiency because we are not training, yet
with torch.no_grad(): 
    # Via the `device` setting, we ensure that the data is loaded onto the same device as the LLM model
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

`train_model_simple` clears gradients, runs a forward pass, backpropagates loss, updates parameters, and periodically records comparable train and validation estimates.

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                        eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], [] #  Initialize lists to track losses and tokens seen
    tokens_seen, global_step = 0, -1
    
    for epoch in range(num_epochs): #  Start the main training loop
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel()
            global_step += 1

            
            if global_step % eval_freq == 0: # Optional evaluation step
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

                
        generate_and_print_sample( #  Print a sample text after each epoch
            model, tokenizer, device, start_context
        )
    return train_losses, val_losses, track_tokens_seen

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval() #  Dropout is disabled during evaluation for stable, reproducible results
    with torch.no_grad(): # Disable gradient tracking, which is not required during evaluation, to reduce the computational overhead
        train_loss = calc_loss_loader(
            train_loader,
            model,
            device,
            num_batches=eval_iter
        )

        val_loss = calc_loss_loader(
            val_loader,
            model,
            device,
            num_batches=eval_iter
        )

    model.train()
    return train_loss, val_loss

In [ ]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
        decoded_text = token_ids_to_text(token_ids, tokenizer)
        print(decoded_text.replace("\n", " "))
     # Compact print format
    model.train()

AdamW provides adaptive updates with decoupled weight decay. The selected hyperparameters are implementation settings, not a claim of optimality.

In [ ]:
import torch

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1) # The .parameters() method returns all trainable weight parameters of the model
num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=1,
    start_context="Every effort moves you", tokenizer=tokenizer
)

The plotting helper remains for a later clean run. Outputs are empty, so no convergence or loss-value claim is made here.

In [ ]:
import matplotlib.pyplot as plt
def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax2 = ax1.twiny() # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0) # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")
    fig.tight_layout()
    plt.show()
    
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

### 6.3 Text Generation

Periodic prompt generation complements loss with a qualitative check, but any model-behavior observation must be regenerated in a controlled run.

In [ ]:
model.to("cpu")
model.eval()

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = generate_text_simple(
model=model,
idx=text_to_token_ids("Every effort moves you", tokenizer),
max_new_tokens=25,
context_size=GPT_CONFIG_124M["context_length"]
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

### 6.4 Sampling Controls

Greedy decoding selects the largest logit. Stochastic decoding samples from an adjusted distribution and can produce different continuations from the same prompt.

In [ ]:
vocab = {
"closer": 0,
"every": 1,
"effort": 2,
"forward": 3,
"inches": 4,
"moves": 5,
"pizza": 6,
"toward": 7,
"you": 8,
}
inverse_vocab = {v: k for k, v in vocab.items()}

In [ ]:
next_token_logits = torch.tensor(
[4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

In [ ]:
probas = torch.softmax(next_token_logits, dim=0)
next_token_id = torch.argmax(probas).item()
print(inverse_vocab[next_token_id])

In [ ]:
torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(inverse_vocab[next_token_id])

Repeated multinomial draws provide an empirical probability check. A fixed seed makes that diagnostic reproducible.

In [ ]:
def print_sampled_tokens(probas):
    torch.manual_seed(123)
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")
print_sampled_tokens(probas)

Temperature rescales logits:

\[
p_i(T)=\operatorname{softmax}(z_i/T).
\]

Lower positive values sharpen the distribution; higher values flatten it. Zero uses a separate greedy branch.

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

temperatures = [1, 0.1, 5] # Original, lower, and higher confidence
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]
x = torch.arange(len(vocab))
bar_width = 0.15
fig, ax = plt.subplots(figsize=(5, 3))
for i, T in enumerate(temperatures):
    rects = ax.bar(x + i * bar_width, scaled_probas[i],
                    bar_width, label=f'Temperature = {T}')
ax.set_ylabel('Probability')
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

Top-k filtering retains the `k` largest logits and replaces the rest with negative infinity before normalization, restricting the candidate set.

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)
print("Top logits:", top_logits)
print("Top positions:", top_pos)

In [ ]:
new_logits = torch.where(
    condition=next_token_logits < top_logits[-1], # Identifies logits less than the minimum in the top 3
     input=torch.tensor(float('-inf')), # Assigns -inf to these lower logits
     other=next_token_logits # Retains the original logits for all other tokens
     )
print(new_logits)

Temperature changes probability shape; top-k changes support. Their behavioral effect should be compared later with fixed prompts and regenerated samples.

In [ ]:
topk_probas = torch.softmax(new_logits, dim=0)
print(topk_probas)

The expanded `generate` helper combines context cropping, optional top-k filtering, greedy or temperature-based selection, and optional end-token stopping.

In [ ]:
def generate(
    model,
    idx,
    max_new_tokens,
    context_size,
    temperature=1.0,
    top_k=None,
    eos_id=None
):
    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):

        # Limit the context to the maximum supported length.
        idx_cond = idx[:, -context_size:]

        # Predicting the next token 
        with torch.no_grad():
            logits = model(idx_cond)

        # Only use the logits for the last token.
        logits = logits[:, -1, :]

        # Optional: Keep only the top k most probable tokens.
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k) #  In this new section, we filter logits with top_k sampling

            # Smallest logit within the top k
            min_val = top_logits[:, -1].unsqueeze(-1)

            # Set all other logits to -infinity
            logits = torch.where(
                logits < min_val,
                torch.tensor(
                    float("-inf"),
                    device=logits.device,
                    dtype=logits.dtype
                ),
                logits
            )

        # Sampling with Temperatur
        if temperature > 0.0: # This is the new section where we apply temperature scaling
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

        # Greedy Decoding without chance
        else: # Carry out greedy next-token selection as before when temperature scaling is disabled
            idx_next = torch.argmax(
                logits,
                dim=-1,
                keepdim=True
            )

        # Abort if the end-of-sequence token was generated
        if eos_id is not None and (idx_next == eos_id).all(): # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Append a new token to the existing sequence
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()

torch.manual_seed(123)

input_ids = text_to_token_ids(
    "Every effort moves you",
    tokenizer
).to(device)

token_ids = generate(
    model=model,
    idx=input_ids,
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print(
    "Output text:\n",
    token_ids_to_text(token_ids.cpu(), tokenizer)
)

### 6.5 Checkpointing

A model `state_dict` stores learned parameters. Resumable training also requires optimizer state because AdamW maintains moving statistics for each parameter.

In [ ]:
"""
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load(CHECKPOINT_DIR / "model.pth"))
model.eval()
"""

In [ ]:
torch.save({
"model_state_dict": model.state_dict(),
"optimizer_state_dict": optimizer.state_dict(),
},
MODEL_OPTIMIZER_PATH
)

Checkpoint files are ignored generated artifacts. Loading is demonstrated locally; no trained model is embedded in this notebook or repository.

In [ ]:
checkpoint = torch.load(MODEL_OPTIMIZER_PATH)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

In [ ]:
# pip install "tensorflow>=2.15.0" "tqdm>=4.66"

## 7. Loading Pretrained GPT-2 Weights

A compatible randomly initialized architecture becomes useful for inference only after training or parameter import. This section maps official GPT-2 artifacts into the local module hierarchy.

> **GPT-2 provenance boundary.** `gpt_download.py`, `assign`, and `load_weights_into_gpt` are adapted from Raschka's pinned revision `33f5b246766464910accf1c70e668811cfc4bf08`. The helper is retrieved at runtime. Model artifacts come from OpenAI and are not stored here. See [`../THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

In [ ]:
import hashlib
import urllib.request


GPT_DOWNLOAD_URL = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/33f5b246766464910accf1c70e668811cfc4bf08/ch05/"
    "01_main-chapter-code/gpt_download.py"
)
GPT_DOWNLOAD_SHA256 = (
    "3e2e95f7c3bf188e16e33e23fc9f0b1d285da55931f15fe12174d32cf10ec076"
)

if not GPT_DOWNLOAD_PATH.exists():
    urllib.request.urlretrieve(GPT_DOWNLOAD_URL, GPT_DOWNLOAD_PATH)

helper_sha256 = hashlib.sha256(GPT_DOWNLOAD_PATH.read_bytes()).hexdigest()
if helper_sha256 != GPT_DOWNLOAD_SHA256:
    raise RuntimeError(
        f"Unexpected SHA-256 for {GPT_DOWNLOAD_PATH}: {helper_sha256}"
    )

The helper downloads checkpoint shards and converts TensorFlow variables into nested Python and NumPy structures. Runtime retrieval avoids embedding another source copy.

In [ ]:
import importlib.util


gpt_download_spec = importlib.util.spec_from_file_location(
    "runtime_gpt_download", GPT_DOWNLOAD_PATH
)
if gpt_download_spec is None or gpt_download_spec.loader is None:
    raise RuntimeError(f"Could not load GPT-2 helper from {GPT_DOWNLOAD_PATH}")

gpt_download_module = importlib.util.module_from_spec(gpt_download_spec)
gpt_download_spec.loader.exec_module(gpt_download_module)
download_and_load_gpt2 = gpt_download_module.download_and_load_gpt2
settings, params = download_and_load_gpt2(
    model_size="124M", models_dir=GPT2_DIR
)


In [ ]:
print("Settings:", settings)
print("Parameter dictionary keys:", params.keys())

GPT-2 sizes differ in embedding width, layer count, and head count. The destination configuration must match the selected artifact before assignment.

In [ ]:
print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)

In [ ]:
model_configs = {
"gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
"gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
"gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
"gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

Compatibility also includes the original context length and projection-bias setting. These values are applied before constructing the destination model.

In [ ]:
model_name = "gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])

In [ ]:
NEW_CONFIG.update({"context_length": 1024})

In [ ]:
NEW_CONFIG.update({"qkv_bias": True})

In [ ]:
gpt = GPTModel(NEW_CONFIG)
gpt.eval()

`assign` rejects shape mismatches before replacing a parameter, converting an otherwise subtle architecture error into an immediate failure.

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

`load_weights_into_gpt` walks every block, splits combined query/key/value arrays, transposes differing storage layouts, and assigns embeddings, projections, normalization parameters, and the output head.

In [ ]:
import numpy as np


def load_weights_into_gpt(gpt, params):
    # Setting the model's positional and token embedding weights to those specified in params.
    gpt.pos_emb.weight = assign(
        gpt.pos_emb.weight,
        params["wpe"]
    )

    gpt.tok_emb.weight = assign(
        gpt.tok_emb.weight,
        params["wte"]
    )

     # Iterate over each transformer block in the model.
    for b in range(len(params["blocks"])):
        block_params = params["blocks"][b]
        block = gpt.trf_blocks[b]

        # The np.split function is used to divide the attention and bias weights into three equal parts for the query,
        # key, and value components.
        q_w, k_w, v_w = np.split(
            block_params["attn"]["c_attn"]["w"],
            3,
            axis=-1
        )

        block.att.W_query.weight = assign(
            block.att.W_query.weight,
            q_w.T
        )

        block.att.W_key.weight = assign(
            block.att.W_key.weight,
            k_w.T
        )

        block.att.W_value.weight = assign(
            block.att.W_value.weight,
            v_w.T
        )

        # Combined Query-, Key- und Value-Bias aufteilen
        q_b, k_b, v_b = np.split(
            block_params["attn"]["c_attn"]["b"],
            3,
            axis=-1
        )

        block.att.W_query.bias = assign(
            block.att.W_query.bias,
            q_b
        )

        block.att.W_key.bias = assign(
            block.att.W_key.bias,
            k_b
        )

        block.att.W_value.bias = assign(
            block.att.W_value.bias,
            v_b
        )

        # Attention Output Projection
        block.att.out_proj.weight = assign(
            block.att.out_proj.weight,
            block_params["attn"]["c_proj"]["w"].T
        )

        block.att.out_proj.bias = assign(
            block.att.out_proj.bias,
            block_params["attn"]["c_proj"]["b"]
        )

        # Feed-Forward-Netzwerk
        block.ff.layers[0].weight = assign(
            block.ff.layers[0].weight,
            block_params["mlp"]["c_fc"]["w"].T
        )

        block.ff.layers[0].bias = assign(
            block.ff.layers[0].bias,
            block_params["mlp"]["c_fc"]["b"]
        )

        block.ff.layers[2].weight = assign(
            block.ff.layers[2].weight,
            block_params["mlp"]["c_proj"]["w"].T
        )

        block.ff.layers[2].bias = assign(
            block.ff.layers[2].bias,
            block_params["mlp"]["c_proj"]["b"]
        )

        # Layer Normalization
        block.norm1.scale = assign(
            block.norm1.scale,
            block_params["ln_1"]["g"]
        )

        block.norm1.shift = assign(
            block.norm1.shift,
            block_params["ln_1"]["b"]
        )

        block.norm2.scale = assign(
            block.norm2.scale,
            block_params["ln_2"]["g"]
        )

        block.norm2.shift = assign(
            block.norm2.shift,
            block_params["ln_2"]["b"]
        )

    # Finale Layer Normalization – außerhalb der Schleife
    gpt.final_norm.scale = assign(
        gpt.final_norm.scale,
        params["g"]
    )

    gpt.final_norm.shift = assign(
        gpt.final_norm.shift,
        params["b"]
    )

    # The original GPT-2 model by OpenAI reused the token embedding weights in the output layer to reduce the
    # total number of parameters, which is a concept known as weight tying.
    gpt.out_head.weight = assign(
        gpt.out_head.weight,
        params["wte"]
    )

A generated continuation after loading checks configuration, mapping, and inference together. Random-versus-pretrained behavior will be documented only after outputs are regenerated.

In [ ]:
load_weights_into_gpt(gpt, params)
gpt.to(device)

In [ ]:
device = next(gpt.parameters()).device

torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids(
        "Every effort moves you",
        tokenizer
    ).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print(
    "Output text:\n",
    token_ids_to_text(token_ids.cpu(), tokenizer)
)

In [ ]:
# Ensure that the pretrained model is on the correct device
gpt.to(device)

# Disable dropout during evaluation
gpt.eval()

# No gradients are required because we are not training the model
with torch.no_grad():
    train_loss = calc_loss_loader(
        train_loader,
        gpt,
        device
    )

    val_loss = calc_loss_loader(
        val_loader,
        gpt,
        device
    )

print(f"Training loss: {train_loss:.6f}")
print(f"Validation loss: {val_loss:.6f}")

## 8. Classification Fine-Tuning

This is an educational transfer-learning experiment using public SMS data. It is distinct from `projects/email-spam-detector/` and does not use private email data or that project's pipeline.

### 8.1 Dataset Preparation

> **Data and implementation provenance.** The UCI SMS Spam Collection is retrieved at runtime under CC BY 4.0. The fine-tuning code closely follows Raschka's pinned implementation. Citations and terms are in [`../THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

In [ ]:
import urllib.request
import zipfile
import os

UCI_SMS_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
SMS_DATA_PATH = SMS_EXTRACTED_DIR / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return
    with urllib.request.urlopen(url) as response: # Downloading the file
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, "r") as zip_ref: # Unzipping the file
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path) # Adding a .tsv file extension
    print(f"File downloaded and saved as {data_file_path}")
    
download_and_unzip_spam_data(
    UCI_SMS_URL, SMS_ZIP_PATH, SMS_EXTRACTED_DIR, SMS_DATA_PATH
)

The downloaded table contains message text and a `ham` or `spam` label. Inspection should stay at schema and aggregate level; raw rows are not retained as output.

In [ ]:
import pandas as pd
df = pd.read_csv(SMS_DATA_PATH, sep="\t", header=None, names=["Label", "Text"])
df # Renders the data frame in a Jupyter notebook. Alternatively, use print(df).
 

In [ ]:
print(df["Label"].value_counts())

The majority class is undersampled to form a balanced educational dataset. This simplifies accuracy interpretation but no longer reflects deployment prevalence.

In [ ]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0] # Count the instances of "spam"
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123) # Randomly sample "ham" instances to match the number of "spam" instances
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]]) # Combine ham "subset" with "spam"
    return balanced_df
    
balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

In [ ]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

A seeded shuffle precedes a 70/10/20 train, validation, and test split, making membership repeatable for the downloaded dataset version.

In [ ]:
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True) # Shuffle the entire DataFrame
    
    train_end = int(len(df) * train_frac) # Calculate split indices
    validation_end = train_end + int(len(df) * validation_frac)

    # Split the DataFrame
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    
    return train_df, validation_df, test_df
train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1) # Test size is implied to be 0.2 as the remainder

Derived CSV splits are ignored runtime artifacts used by the dataset class; row-level files remain outside version control.

In [ ]:
train_df.to_csv(TRAIN_CSV_PATH, index=None)
validation_df.to_csv(VALIDATION_CSV_PATH, index=None)
test_df.to_csv(TEST_CSV_PATH, index=None)

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

`SpamDataset` tokenizes once, truncates to the selected width, pads with GPT-2's end-of-text ID, and returns a fixed-length token tensor plus an integer label.

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset


class SpamDataset(Dataset):
    def __init__(
        self,
        csv_file,
        tokenizer,
        max_length=None,
        pad_token_id=50256
    ):
        self.data = pd.read_csv(csv_file)

        # Pre-tokenize texts
        self.encoded_texts = [
            tokenizer.encode(text)
            for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length

            # Truncate sequences if they are longer than max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text
            + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]

        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0

        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)

            if encoded_length > max_length:
                max_length = encoded_length

        return max_length

In [ ]:
train_dataset = SpamDataset(
csv_file=TRAIN_CSV_PATH,
max_length=None,
tokenizer=tokenizer
)

In [ ]:
print(train_dataset.max_length)

In [ ]:
val_dataset = SpamDataset(
csv_file=VALIDATION_CSV_PATH,
max_length=train_dataset.max_length,
tokenizer=tokenizer
)
test_dataset = SpamDataset(
csv_file=TEST_CSV_PATH,
max_length=train_dataset.max_length,
tokenizer=tokenizer
)

Validation and test data reuse the training maximum length. Their loaders are deterministic, while the training loader shuffles samples.

In [ ]:
from torch.utils.data import DataLoader

# This setting ensures compatibility with most computers
num_workers = 0

batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False,
)

In [ ]:
for input_batch, target_batch in train_loader:
    pass
print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

In [ ]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

### 8.2 Adapting GPT for Classification

The experiment starts from pretrained GPT-2-compatible parameters rather than random initialization.

In [ ]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": True        # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {
        "emb_dim": 768,
        "n_layers": 12,
        "n_heads": 12
    },
    "gpt2-medium (355M)": {
        "emb_dim": 1024,
        "n_layers": 24,
        "n_heads": 16
    },
    "gpt2-large (774M)": {
        "emb_dim": 1280,
        "n_layers": 36,
        "n_heads": 20
    },
    "gpt2-xl (1558M)": {
        "emb_dim": 1600,
        "n_layers": 48,
        "n_heads": 25
    }
}

# Add the architecture settings of the selected model
# to the general GPT configuration
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

# Ensure that the padded dataset sequences do not exceed
# the model's maximum context length
assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"Dataset length {train_dataset.max_length} exceeds model's context "
    f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with "
    f"`max_length={BASE_CONFIG['context_length']}`"
)

In [ ]:
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir=GPT2_DIR)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

In [ ]:
text_1 = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

In [ ]:
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

The vocabulary projection is replaced by a two-logit head. Existing parameters are frozen first; the new head is trainable by construction.

In [ ]:
print(model)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

The final transformer block and final normalization are then unfrozen, adapting high-level representations while limiting compute on the small dataset.

In [ ]:
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(
    
in_features=BASE_CONFIG["emb_dim"],
    out_features=num_classes
)

In [ ]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True

In [ ]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape) # shape: (batch_size, num_tokens)

Classification uses final-position logits. Under causal attention, that position can aggregate all preceding non-padding tokens.

In [ ]:
with torch.no_grad():
    outputs = model(inputs)
print("Outputs:\n", outputs)
print("Outputs dimensions:", outputs.shape) # shape: (batch_size, num_tokens,num_classes)

In [ ]:
print("Last output token:", outputs[:, -1, :])

In [ ]:
print("Last output token:", outputs[:, -1, :])

In [ ]:
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print("Class label:", label.item())

In [ ]:
logits = outputs[:, -1, :]
label = torch.argmax(logits)
print("Class label:", label.item())

### 8.3 Training and Evaluation

Accuracy is argmax agreement over two logits. Evaluation mode and disabled gradients make the estimate deterministic for fixed parameters.

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()

    correct_predictions = 0
    num_examples = 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            with torch.no_grad():
                # Logits of the last output token
                logits = model(input_batch)[:, -1, :]

            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]

            correct_predictions += (
                predicted_labels == target_batch
            ).sum().item()

        else:
            break

    return correct_predictions / num_examples

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Classification cross-entropy uses one two-class vector per message instead of one vocabulary vector per token.

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
     # Logits of last output token
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0

    if len(data_loader) == 0:
        return float("nan")

    elif num_batches is None:
        num_batches = len(data_loader)

    else:
        # A: Ensure number of batches doesn't exceed
        # the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
                input_batch,
                target_batch,
                model,
                device
            )
            total_loss += loss.item()

        else:
            break

    return total_loss / num_batches


model.eval()

with torch.no_grad():
    # B: Disable gradient tracking for efficiency
    # because we are not training yet
    train_loss = calc_loss_loader(
        train_loader,
        model,
        device,
        num_batches=5
    )

    val_loss = calc_loss_loader(
        val_loader,
        model,
        device,
        num_batches=5
    )

    test_loss = calc_loss_loader(
        test_loader,
        model,
        device,
        num_batches=5
    )


print(f"Training loss:   {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss:       {test_loss:.3f}")

The fine-tuning loop updates only unfrozen parameters and records periodic loss and accuracy estimates. Metrics will be regenerated during reproducibility work.

In [ ]:
def train_classifier_simple(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs,
    eval_freq,
    eval_iter,
    tokenizer
):
    # Initialize lists to track losses and examples seen
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    examples_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):

        # A: Set model to training mode
        model.train()

        for input_batch, target_batch in train_loader:

            # B: Reset loss gradients from the previous batch iteration
            optimizer.zero_grad()

            loss = calc_loss_batch(
                input_batch,
                target_batch,
                model,
                device
            )

            # C: Calculate loss gradients
            loss.backward()

            # D: Update model weights using loss gradients
            optimizer.step()

            # E: Track examples instead of tokens
            examples_seen += input_batch.shape[0]

            global_step += 1

            # F: Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model,
                    train_loader,
                    val_loader,
                    device,
                    eval_iter
                )

                train_losses.append(train_loss)
                val_losses.append(val_loss)

                print(
                    f"Ep {epoch + 1} "
                    f"(Step {global_step:06d}): "
                    f"Train loss {train_loss:.3f}, "
                    f"Val loss {val_loss:.3f}"
                )

        # G: Calculate accuracy after each epoch
        train_accuracy = calc_accuracy_loader(
            train_loader,
            model,
            device,
            num_batches=eval_iter
        )

        val_accuracy = calc_accuracy_loader(
            val_loader,
            model,
            device,
            num_batches=eval_iter
        )

        print(
            f"Training accuracy: {train_accuracy * 100:.2f}% | ",
            end=""
        )
        print(
            f"Validation accuracy: {val_accuracy * 100:.2f}%"
        )

        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return (
        train_losses,
        val_losses,
        train_accs,
        val_accs,
        examples_seen
    )

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device,
num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
import time

start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_epochs = 5

train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
    tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
import matplotlib.pyplot as plt


def plot_values(
    epochs_seen,
    examples_seen,
    train_values,
    val_values,
    label="loss"
):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # A: Plot training and validation values against epochs
    ax1.plot(
        epochs_seen,
        train_values,
        label=f"Training {label}"
    )

    ax1.plot(
        epochs_seen,
        val_values,
        linestyle="-.",
        label=f"Validation {label}"
    )

    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    # B: Create a second x-axis for examples seen
    ax2 = ax1.twiny()

    # Invisible plot for aligning the second x-axis ticks
    ax2.plot(
        examples_seen,
        train_values,
        alpha=0
    )

    ax2.set_xlabel("Examples seen")

    # C: Adjust layout to make room for both x-axes
    fig.tight_layout()

    plt.savefig(PLOT_DIR / f"{label}-plot.pdf")
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))
plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)
print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

The inference helper repeats training-time tokenization, truncation, padding, final-position selection, and argmax. Optional checkpoints remain ignored artifacts.

In [ ]:
import torch


def classify_review(
    text,
    model,
    tokenizer,
    device,
    max_length=None,
    pad_token_id=50256
):
    model.eval()

    # A: Prepare inputs for the model
    input_ids = tokenizer.encode(text)

    # Determine the maximum context length supported by the model
    supported_context_length = model.pos_emb.weight.shape[0]

    # Use the model's supported context length if max_length is not specified
    if max_length is None:
        max_length = supported_context_length

    # Ensure max_length does not exceed the model's context length
    max_length = min(max_length, supported_context_length)

    # B: Truncate the sequence if it is too long
    input_ids = input_ids[:max_length]

    # C: Pad the sequence to max_length
    input_ids += [pad_token_id] * (max_length - len(input_ids))

    # D: Convert token IDs to a tensor and add a batch dimension
    input_tensor = torch.tensor(
        input_ids,
        device=device
    ).unsqueeze(0)

    # E: Model inference without gradient tracking
    with torch.no_grad():

        # F: Use the logits of the last output token
        logits = model(input_tensor)[:, -1, :]

    predicted_label = torch.argmax(
        logits,
        dim=-1
    ).item()

    # G: Return the classified result
    return "spam" if predicted_label == 1 else "not spam"

In [ ]:
text_1 = (
"You are a winner you have been specially"
" selected to receive $1000 cash or a $2000 award."
)
print(classify_review(
text_1, model, tokenizer, device, max_length=train_dataset.max_length
))

In [ ]:
text_2 = (
"Hey, just wanted to check if we're still on"
" for dinner tonight? Let me know!"
)
print(classify_review(
text_2, model, tokenizer, device, max_length=train_dataset.max_length
))

In [ ]:
torch.save(model.state_dict(), REVIEW_CLASSIFIER_PATH)

In [ ]:
model_state_dict = torch.load(REVIEW_CLASSIFIER_PATH)
model.load_state_dict(model_state_dict)

## 9. Instruction Fine-Tuning

### 9.1 Instruction Format

Each runtime-loaded record contains an instruction, optional input, and target response. No local JSON copy is redistributed, and its artifact-level provenance caveat is documented in [`../THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

> **Adapted implementation.** Formatting, dataset, collation, fine-tuning, and evaluation helpers closely follow Raschka's pinned instruction pipeline.

In [ ]:
import json
import hashlib
import urllib.request


def download_and_load_file(file_path, url, expected_sha256):
    file_path = Path(file_path)

    # A: Skip download if file was already downloaded
    if not file_path.exists():
        with urllib.request.urlopen(url) as response:
            file_path.write_bytes(response.read())

    actual_sha256 = hashlib.sha256(file_path.read_bytes()).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(
            f"Unexpected SHA-256 for {file_path}: {actual_sha256}"
        )

    with file_path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    return data


INSTRUCTION_DATA_URL = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/33f5b246766464910accf1c70e668811cfc4bf08/ch07/01_main-chapter-code/"
    "instruction-data.json"
)
INSTRUCTION_DATA_SHA256 = (
    "7e09a9af353722ca15c18bc580f093f9c39d3178335a75177319cb1e1d163944"
)

data = download_and_load_file(
    INSTRUCTION_DATA_PATH, INSTRUCTION_DATA_URL, INSTRUCTION_DATA_SHA256
)

print("Number of entries:", len(data))

In [ ]:
print("Example entry:\n", data[50])

In [ ]:
print("Another example entry:\n", data[999])

`format_input` creates stable instruction and response boundaries and omits the optional input block when it is empty.

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

In [ ]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

In [ ]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)

Records are divided into train, test, and validation subsets before encoding. Raw entries are not preserved as notebook output.

In [ ]:
train_portion = int(len(data) * 0.85) # 85% for training
test_portion = int(len(data) * 0.1) # 10% for testing
val_portion = len(data) - train_portion - test_portion # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

In [ ]:
## Organizing Data into training batches

### 9.2 Dataset and Collation

`InstructionDataset` combines formatted prompt and response, tokenizes the training sequence, and stores variable-length ID lists so padding can occur per batch.

In [ ]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []

        # A: Pre-tokenize all texts
        for entry in data:
            instruction_plus_input = format_input(entry)

            response_text = (
                f"\n\n### Response:\n"
                f"{entry['output']}"
            )

            full_text = instruction_plus_input + response_text

            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))


Dynamic collation pads to the longest item in each batch plus one token, reducing unnecessary work relative to one dataset-wide maximum.

In [ ]:
import torch


def custom_collate_draft_1(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # A: Find the longest sequence in the batch
    # Add 1 because an additional padding token is appended to each item
    batch_max_length = max(len(item) + 1 for item in batch)

    inputs_lst = []

    for item in batch:
        # B: Pad and prepare inputs
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = new_item + [pad_token_id] * (
            batch_max_length - len(new_item)
        )

        # C: Remove the extra padded token added earlier
        inputs = torch.tensor(padded[:-1])

        inputs_lst.append(inputs)

    # D: Convert the list of inputs to a tensor
    # and transfer it to the target device
    inputs_tensor = torch.stack(inputs_lst).to(device)

    return inputs_tensor

In [ ]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
print(custom_collate_draft_1(batch))

Inputs omit the final padded token and targets omit the first token, creating the same one-position shift used in language-model pretraining.

In [ ]:
import torch


def custom_collate_draft_2(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # Find the longest sequence in the batch
    # and reserve space for one additional token
    batch_max_length = max(len(item) + 1 for item in batch)

    inputs_lst = []
    targets_lst = []

    for item in batch:
        new_item = item.copy()

        # Add one padding token to the end of the sequence
        new_item += [pad_token_id]

        # Pad all sequences to the same length
        padded = new_item + [pad_token_id] * (
            batch_max_length - len(new_item)
        )

        # A: Truncate the last token for the inputs
        inputs = torch.tensor(padded[:-1])

        # B: Shift the targets one position ahead
        targets = torch.tensor(padded[1:])

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert the lists into batch tensors
    # and transfer them to the target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor


inputs, targets = custom_collate_draft_2(batch)

print("Inputs:")
print(inputs)

print("\nTargets:")
print(targets)

### 9.3 Target Masking

Most padding targets become `ignore_index=-100`, which cross-entropy excludes. One end-of-text target remains so termination can still be learned.

In [ ]:
import torch


def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # Determine the maximum sequence length in the batch.
    # Add one extra token because the targets are shifted by one position.
    batch_max_length = max(len(item) + 1 for item in batch)

    inputs_lst = []
    targets_lst = []

    for item in batch:
        new_item = item.copy()

        # Add one padding token to the end of the sequence
        new_item += [pad_token_id]

        # Pad all sequences to the same length
        padded = new_item + [pad_token_id] * (
            batch_max_length - len(new_item)
        )

        # Truncate the last token for the inputs
        inputs = torch.tensor(padded[:-1])

        # Shift the targets one position ahead
        targets = torch.tensor(padded[1:])

        # A: Replace all but the first padding token in the targets
        # with ignore_index so they do not contribute to the loss
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).flatten()

        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # B: Optionally truncate inputs and targets
        # to the maximum permitted sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert the lists into batch tensors
    # and transfer them to the selected device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [ ]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

Small synthetic loss checks isolate the effect of ignored positions without exposing any external dataset record.

In [ ]:
logits_1 = torch.tensor(
[[-1.0, 1.0], # predictions for 1st token
[-0.5, 1.5]] # predictions for 2nd token
)
targets_1 = torch.tensor([0, 1]) # Correct token indices to generate
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

In [ ]:
logits_2 = torch.tensor(
[[-1.0, 1.0],
[-0.5, 1.5],
[-0.5, 1.5]] # New 3rd token ID prediction

)
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)


In [ ]:
targets_3 = torch.tensor([0, 1, -100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)


The final collate function also applies an optional maximum length and device transfer. `functools.partial` fixes those policies for all dataloaders.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# if torch.backends.mps.is_available(): # Uncomment these two lines to use the GPU on an Apple Silicon chip
# device = torch.device("mps") # Uncomment these two lines to use the GPU on an Apple Silicon chip
print("Device:", device)

In [ ]:
from functools import partial

customized_collate_fn = partial(custom_collate_fn, device=device,
allowed_max_length=1024)

In [ ]:
import torch
from torch.utils.data import DataLoader


# A: You can try increasing this number if parallel Python
# processes are supported by your operating system
num_workers = 0

batch_size = 2

# Set the random seed for reproducible shuffling
torch.manual_seed(123)


# Training dataset and data loader
train_dataset = InstructionDataset(
    train_data,
    tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)


# Validation dataset and data loader
val_dataset = InstructionDataset(
    val_data,
    tokenizer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)


# Test dataset and data loader
test_dataset = InstructionDataset(
    test_data,
    tokenizer
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [ ]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)


### 9.4 Fine-Tuning

A pretrained compatible model is evaluated on a validation prompt before updates, establishing the workflow for a later regenerated comparison.

In [ ]:



BASE_CONFIG = {
    "vocab_size": 50257,      # Vocabulary size
    "context_length": 1024,   # Context length
    "drop_rate": 0.0,         # Dropout rate
    "qkv_bias": True          # Query-key-value bias
}


model_configs = {
    "gpt2-small (124M)": {
        "emb_dim": 768,
        "n_layers": 12,
        "n_heads": 12
    },
    "gpt2-medium (355M)": {
        "emb_dim": 1024,
        "n_layers": 24,
        "n_heads": 16
    },
    "gpt2-large (774M)": {
        "emb_dim": 1280,
        "n_layers": 36,
        "n_heads": 20
    },
    "gpt2-xl (1558M)": {
        "emb_dim": 1600,
        "n_layers": 48,
        "n_heads": 25
    }
}


# Select the GPT-2 model size
CHOOSE_MODEL = "gpt2-small (124M)"


# Add the selected model configuration to BASE_CONFIG
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])


# Extract the model size required by download_and_load_gpt2()
# Example: "gpt2-medium (355M)" becomes "355M"
model_size = (
    CHOOSE_MODEL
    .split(" ")[-1]
    .lstrip("(")
    .rstrip(")")
)


# Download and load the original OpenAI GPT-2 parameters
settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir=GPT2_DIR
)


# Initialize the GPT model architecture
model = GPTModel(BASE_CONFIG)


# Transfer the pretrained OpenAI GPT-2 weights into the model
load_weights_into_gpt(
    model,
    params
)


# Set the model to evaluation mode
model.eval()

In [ ]:
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(input_text)


In [ ]:
token_ids = generate(
model=model,
idx=text_to_token_ids(input_text, tokenizer),
max_new_tokens=35,
context_size=BASE_CONFIG["context_length"],
eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)


In [ ]:
response_text = generated_text[len(input_text):].strip()
print(response_text)


Instruction fine-tuning still uses ordinary next-token cross-entropy, with loss computed across the full formatted instruction, input, and response sequence. The collator sets repeated padding targets after the first padding/EOT target to `ignore_index=-100`; instruction and input tokens are not masked from the loss. The small model configuration reflects this project's limited compute, not a hardware recommendation.

In [ ]:
import torch.nn.functional as F


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)

    # Vorhersagen für alle Tokenpositionen:
    # [batch_size, sequence_length, vocab_size]
    logits = model(input_batch)

    # Für cross_entropy umformen:
    # logits:  [batch_size * sequence_length, vocab_size]
    # targets: [batch_size * sequence_length]
    loss = F.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten()
    )

    return loss

In [ ]:
model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
import time

start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)
num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)


### 9.5 Response Generation

The generation loop formats each test instruction, creates a continuation, removes the prompt prefix, and first stores each extracted response in the in-memory test data. The enriched data is then serialized to the generated runtime file `instruction-data-with-response.json`; it is not a tracked repository asset. Old response samples are not reproduced.

In [ ]:
torch.manual_seed(123)

for entry in test_data[:3]: # Iterate over the first 3 test set samples
    input_text = format_input(entry)
    token_ids = generate( # Use the generate function we  imported 
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].replace("### Response:",
"").strip()
    
    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("-------------------------------------")

In [ ]:
from tqdm import tqdm

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)
    
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].replace("### Response:",
"").strip()
    test_data[i]["model_response"] = response_text
    
with INSTRUCTION_RESPONSES_PATH.open("w", encoding="utf-8") as file:
    json.dump(test_data, file, indent=4) # "indent" for pretty-printing

In [ ]:
print(test_data[0])

In [ ]:
import re
# Remove white spaces and parentheses from file name
file_name = f"{re.sub(r'[ ()]', '', CHOOSE_MODEL) }-sft.pth"
checkpoint_path = CHECKPOINT_DIR / file_name
torch.save(model.state_dict(), checkpoint_path)
print(f"Model saved as {checkpoint_path}")


In [ ]:
model.load_state_dict(torch.load(SFT_CHECKPOINT_PATH))

### 9.6 Evaluation

The evaluator is optional post-processing backed by a local external service. It runs only when `RUN_EXTERNAL_EVALUATION` is set to `True`; Ollama availability and evaluator model version are then part of the reproducibility boundary.

In [ ]:
import psutil

RUN_EXTERNAL_EVALUATION = False
EVALUATOR_MODEL = "qwen3.5:4b"


def check_if_running(process_name):
    process_name = process_name.lower()

    for proc in psutil.process_iter(["name"]):
        name = proc.info.get("name") or ""

        if process_name in name.lower():
            return True

    # Erst zurückgeben, nachdem alle Prozesse geprüft wurden
    return False


if RUN_EXTERNAL_EVALUATION:
    ollama_running = check_if_running("ollama")

    if not ollama_running:
        raise RuntimeError(
            "Ollama not running. Launch Ollama before proceeding."
        )

    print("Ollama running:", ollama_running)
else:
    print("External Ollama evaluation skipped (RUN_EXTERNAL_EVALUATION=False).")

In [ ]:
import json
from tqdm import tqdm

if RUN_EXTERNAL_EVALUATION:
    with INSTRUCTION_RESPONSES_PATH.open("r", encoding="utf-8") as file:
        test_data = json.load(file)
    
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

`query_model` sends deterministic requests with explicit error handling. The evaluator receives the instruction, reference, and candidate response and returns a requested numeric judgment.

In [ ]:
import json
import urllib.error
import urllib.request


def query_model(
    prompt,
    evaluator_model=EVALUATOR_MODEL,
    url="http://localhost:11434/api/chat"
):
    data = {
        "model": evaluator_model,

        # Return one complete JSON response instead of a stream
        "stream": False,

        # Disable separate reasoning output
        "think": False,

        # Keep the model loaded for subsequent evaluation requests
        "keep_alive": "10m",

        # Generation settings
        "options": {
            "seed": 123,
            "temperature": 0,
            "num_predict": 256
        },

        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    payload = json.dumps(data).encode("utf-8")

    request = urllib.request.Request(
        url=url,
        data=payload,
        method="POST",
        headers={
            "Content-Type": "application/json"
        }
    )

    try:
        with urllib.request.urlopen(request, timeout=180) as response:
            response_json = json.load(response)

        return response_json["message"]["content"].strip()

    except urllib.error.HTTPError as error:
        error_message = error.read().decode("utf-8")
        raise RuntimeError(
            f"Ollama returned HTTP {error.code}: {error_message}"
        ) from error

    except urllib.error.URLError as error:
        raise RuntimeError(
            "The Ollama API is not reachable at "
            "http://localhost:11434. Make sure Ollama is running."
        ) from error

In [ ]:
if RUN_EXTERNAL_EVALUATION:
    result = query_model(
        "What do Llamas eat?", evaluator_model=EVALUATOR_MODEL
    )
    print(result)

In [ ]:
if RUN_EXTERNAL_EVALUATION:
    for entry in test_data[:3]:
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry['model_response']}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
        )
        print("\nDataset response:")
        print(">>", entry['output'])
        print("\nModel response:")
        print(">>", entry["model_response"])
        print("\nScore:")
        print(">>", query_model(prompt))
        print("\n-------------------------")


`generate_model_scores` validates numeric responses across the test set. Automated scores are model-dependent proxies; future reporting should identify the evaluator version and pair aggregates with limited human review.

In [ ]:
def generate_model_scores(
    json_data,
    json_key,
    evaluator_model=EVALUATOR_MODEL
):
    scores = []

    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}` "
            f"on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )

        score = query_model(
            prompt=prompt,
            evaluator_model=evaluator_model
        )

        try:
            scores.append(int(score.strip()))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores

No prior score is retained. Aggregate evaluation, example selection, and limitations will be documented only after a controlled rerun.

In [ ]:
if RUN_EXTERNAL_EVALUATION:
    scores = generate_model_scores(
        test_data, "model_response", evaluator_model=EVALUATOR_MODEL
    )
    print(f"Number of scores: {len(scores)} of {len(test_data)}")
    print(f"Average score: {sum(scores)/len(scores):.2f}\n")


## 10. Key Takeaways

- Shifting one token sequence creates next-token supervision.
- Embedding width is the interface shared by attention, residual paths, and stacked blocks.
- Query, key, and value projections separate matching from information transfer.
- Causal masking makes parallel attention autoregressive.
- Multi-head attention changes internal decomposition while preserving external width.
- Pre-normalized residual paths make deep composition practical.
- One transformer backbone supports language modeling, classification, and instruction response training.
- Temperature and top-k alter generation without changing model parameters.

## 11. Limitations and Next Steps

This is an educational GPT-2-era implementation, not a production LLM stack. Training is small-scale, compute is limited, external data and model artifacts are runtime dependencies, and fresh-clone smoke testing has passed on Windows with Python 3.12.10 within the documented non-training scope. The code favors transparency over throughput and omits modern efficiency improvements.

Useful next steps are independent experiments: pin dependencies and complete a clean run; regenerate a compact evidence set; add original tensor-flow visualizations; compare sampling settings on fixed prompts; run controlled context, layer-unfreezing, or attention ablations; compare selected modern components with this baseline; and profile memory and runtime before optimizing code.